# 🔍 05 — Grad-CAM Visualization

Generate visual explanations using Grad-CAM:
- Heatmaps showing which regions influenced predictions
- Comparison across all three models
- Overlay visualizations

In [ ]:
import sys
sys.path.append('..')

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import config
from src.models.model_factory import ModelFactory
from src.utils.gradcam import GradCAM

%matplotlib inline
plt.style.use('dark_background')
print('✓ Libraries loaded')

## 1. Load Models

In [ ]:
models = {}
for name in ModelFactory.list_models():
    try:
        model = ModelFactory.create(name)
        model.load()
        models[name] = model
        print(f'✓ Loaded: {name}')
    except Exception as e:
        print(f'✗ {name}: {e}')

print(f'\nLoaded {len(models)} model(s)')

## 2. Select Test Images
Find test images to visualize.

In [ ]:
test_images = []
test_dir = config.TEST_DIR

for class_dir in sorted(test_dir.iterdir()):
    if class_dir.is_dir():
        images = list(class_dir.glob('*.png')) + list(class_dir.glob('*.jpg'))
        if images:
            test_images.append(str(images[0]))
            print(f'  {class_dir.name}: {images[0].name}')

if not test_images:
    print('No test images found. Creating a synthetic demo image...')
    from PIL import Image
    demo = np.random.randint(50, 200, (300, 300, 3), dtype=np.uint8)
    demo_path = str(config.DATA_DIR / 'gradcam_demo.png')
    Image.fromarray(demo).save(demo_path)
    test_images = [demo_path]

print(f'\n{len(test_images)} test image(s) selected')

## 3. Generate Grad-CAM for Each Model

In [ ]:
for img_path in test_images:
    print(f'\nImage: {Path(img_path).name}')
    print('-' * 40)
    
    n_models = len(models)
    if n_models == 0:
        print('No models loaded.')
        continue
    
    fig, axes = plt.subplots(n_models, 3, figsize=(12, 4 * n_models))
    if n_models == 1:
        axes = [axes]
    
    for idx, (name, model) in enumerate(models.items()):
        cam = GradCAM(model.model, model_name=name)
        
        try:
            heatmap, overlay, prob = cam.generate(img_path)
            pred_class = 'Cancerous' if prob >= 0.5 else 'Normal'
            confidence = prob if prob >= 0.5 else (1 - prob)
            
            # Original
            from tensorflow.keras.preprocessing.image import load_img, img_to_array
            orig = load_img(img_path, target_size=model.img_size)
            axes[idx][0].imshow(orig)
            axes[idx][0].set_title(f'{name.upper()} — Original', fontweight='bold')
            
            # Heatmap
            axes[idx][1].imshow(heatmap, cmap='jet')
            axes[idx][1].set_title('Grad-CAM Heatmap', fontweight='bold')
            
            # Overlay
            axes[idx][2].imshow(overlay)
            color = '#00b894' if pred_class == 'Normal' else '#e17055'
            axes[idx][2].set_title(
                f'{pred_class} ({confidence*100:.1f}%)',
                fontweight='bold', color=color
            )
            
            print(f'  {name.upper()}: {pred_class} ({confidence*100:.1f}%)')
            
        except Exception as e:
            print(f'  {name}: Error — {e}')
            for ax in axes[idx]:
                ax.text(0.5, 0.5, 'Error', ha='center', va='center',
                       transform=ax.transAxes, color='red')
        
        for ax in axes[idx]:
            ax.axis('off')
    
    fig.suptitle(f'Grad-CAM Comparison — {Path(img_path).name}',
                fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

## 4. Save Full Visualizations

In [ ]:
for img_path in test_images:
    for name, model in models.items():
        cam = GradCAM(model.model, model_name=name)
        try:
            output = cam.save_visualization(img_path)
            print(f'Saved: {output}')
        except Exception as e:
            print(f'Error ({name}): {e}')

---
**Done!** All notebooks have been completed. You can now:
1. Deploy the web application: `uvicorn app.main:app --reload`
2. Train with CLI: `python -m src.training.trainer --model resnet50`